In [1]:
import torch
from torch import nn
import torch.nn.functional as F

from models.src.fpn import FPN
from models.src.backbone import Backbone, DetBackbone

/Users/albertomarengo/Bitstrapped-Project/my_yolo/yol-venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [22]:
# input = torch.randn(1, 3, 640, 640)
input = torch.randn(1, 3, 300, 300)

In [23]:
backbone = Backbone('resnet101', True, True)

In [24]:
back_out = backbone(input)

In [25]:
back_out.keys()

odict_keys(['0', '1', '2', '3'])

In [26]:
for layer in back_out:
    print(layer, back_out[layer].shape)

0 torch.Size([1, 256, 75, 75])
1 torch.Size([1, 512, 38, 38])
2 torch.Size([1, 1024, 19, 19])
3 torch.Size([1, 2048, 10, 10])


In [11]:
det_backbone = DetBackbone(
    name="resnet101",
    train_backbone=True,
    returned_layers=[1, 2, 3, 4],
)
det_back_out = det_backbone(input)

/Users/albertomarengo/Bitstrapped-Project/my_yolo/yol-venv/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet101_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet101_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Freezing conv1.weight
Freezing bn1.weight
Freezing bn1.bias
Freezing layer1.0.conv1.weight
Freezing layer1.0.bn1.weight
Freezing layer1.0.bn1.bias
Freezing layer1.0.conv2.weight
Freezing layer1.0.bn2.weight
Freezing layer1.0.bn2.bias
Freezing layer1.0.conv3.weight
Freezing layer1.0.bn3.weight
Freezing layer1.0.bn3.bias
Freezing layer1.0.downsample.0.weight
Freezing layer1.0.downsample.1.weight
Freezing layer1.0.downsample.1.bias
Freezing layer1.1.conv1.weight
Freezing layer1.1.bn1.weight
Freezing layer1.1.bn1.bias
Freezing layer1.1.conv2.weight
Freezing layer1.1.bn2.weight
Freezing layer1.1.bn2.bias
Freezing layer1.1.conv3.weight
Freezing layer1.1.bn3.weight
Freezing layer1.1.bn3.bias
Freezing layer1.2.conv1.weight
Freezing layer1.2.bn1.weight
Freezing layer1.2.bn1.bias
Freezing layer1.2.conv2.weight
Freezing layer1.2.bn2.weight
Freezing layer1.2.bn2.bias
Freezing layer1.2.conv3.weight
Freezing layer1.2.bn3.weight
Freezing layer1.2.bn3.bias
Freezing fc.weight
Freezing fc.bias


In [12]:
det_backbone.num_channels

[256, 512, 1024, 2048]

In [13]:
for layer in det_back_out:
    print(layer, det_back_out[layer].shape)

0 torch.Size([1, 256, 75, 75])
1 torch.Size([1, 512, 38, 38])
2 torch.Size([1, 1024, 19, 19])
3 torch.Size([1, 2048, 10, 10])


In [27]:
fpn = FPN(channels_out=128, block_expansion=4)

In [28]:
fpn_inputs = (back_out[layer] for layer in back_out)
fpn_outputs = fpn(fpn_inputs)

In [29]:
for out in fpn_outputs:
    print(out.shape)

torch.Size([1, 128, 75, 75])
torch.Size([1, 128, 38, 38])
torch.Size([1, 128, 19, 19])
torch.Size([1, 128, 10, 10])


In [50]:
from models.src.fpn import Lateral_Connection

class FPNPAN(nn.Module):

    def __init__(self, channels_out=64, backbone_out_channels = [256, 512, 1024, 2048], **kwargs):

        super().__init__(**kwargs)

        self.fpn_layers_lat_conns = []
        self.fpn_layers_3x3 = []
        self.pan_layers_lat_conns = []
        self.pan_layers_3x3 = []

        start = len(backbone_out_channels)-1
        for i in range(start, -1, -1):
            if i != start:
                self.fpn_layers_lat_conns.append(Lateral_Connection(backbone_out_channels[i], channels_out))
                self.fpn_layers_3x3.append(nn.Conv2d(channels_out, channels_out, kernel_size=3, stride=1, padding=1))
                self.pan_layers_lat_conns.append(Lateral_Connection(channels_out, channels_out, kernel_size=3, padding=1))
            else:
                self.fpn_layer_1x1 = nn.Conv2d(backbone_out_channels[i], channels_out, kernel_size=1, stride=1, padding=0)
                self.pan_last_layer_3x3 = nn.Conv2d(channels_out, channels_out, kernel_size=3, stride=1, padding=1)

            self.pan_layers_3x3.append(nn.Conv2d(channels_out, channels_out, kernel_size=3, stride=1, padding=1))

        self.fpn_layers_lat_conns = nn.ModuleList(self.fpn_layers_lat_conns)
        self.fpn_layers_3x3 = nn.ModuleList(self.fpn_layers_3x3)
        self.pan_layers_lat_conns = nn.ModuleList(self.pan_layers_lat_conns)
        self.pan_layers_3x3 = nn.ModuleList(self.pan_layers_3x3)


    def forward(self, inputs):

        # FPN
        fpn_outputs = []
        latest = self.fpn_layer_1x1(inputs[-1])
        fpn_outputs.append(latest)

        start = len(inputs) - 2
        for idx in range(start, -1, -1):
            latest = self.fpn_layers_lat_conns[start-idx]((latest, inputs[idx]))
            additional_3x3 = F.relu(self.fpn_layers_3x3[start-idx](latest), inplace=True)
            fpn_outputs.append(additional_3x3)

        # PAN
        outputs = []
        pan = F.relu(self.pan_last_layer_3x3(fpn_outputs[-1]), inplace=True)
        outputs.append(pan)
        for idx in range(len(fpn_outputs)-1, 0, -1):
            pan = self.pan_layers_lat_conns[idx-1]((fpn_outputs[idx], fpn_outputs[idx-1]))
            latest = F.relu(self.pan_layers_3x3[idx](pan), inplace=True)
            outputs.append(latest)

        return outputs

In [51]:
alt_fpn = FPNPAN(channels_out=128, backbone_out_channels=det_backbone.num_channels)

In [52]:
fpn_inputs = [det_back_out[layer] for layer in det_back_out]
for input in fpn_inputs:
    print(input.shape)

torch.Size([1, 256, 75, 75])
torch.Size([1, 512, 38, 38])
torch.Size([1, 1024, 19, 19])
torch.Size([1, 2048, 10, 10])


In [53]:
alt_fpn.fpn_layer_1x1

Conv2d(2048, 128, kernel_size=(1, 1), stride=(1, 1))

In [54]:
alt_fpn.fpn_layers_lat_conns

ModuleList(
  (0): Lateral_Connection(
    (conv1x1): Conv2d(1024, 128, kernel_size=(1, 1), stride=(1, 1))
  )
  (1): Lateral_Connection(
    (conv1x1): Conv2d(512, 128, kernel_size=(1, 1), stride=(1, 1))
  )
  (2): Lateral_Connection(
    (conv1x1): Conv2d(256, 128, kernel_size=(1, 1), stride=(1, 1))
  )
)

In [55]:
fpn_alt_outputs = alt_fpn(fpn_inputs)

In [56]:
for out in fpn_alt_outputs:
    print(out.shape)

torch.Size([1, 128, 75, 75])
torch.Size([1, 128, 38, 38])
torch.Size([1, 128, 19, 19])
torch.Size([1, 128, 10, 10])


In [ ]:
test = Lateral_Connection(128, 128, kernel_size=3, padding=1)
(test((fpn_alt_outputs[-1], fpn_alt_outputs[-2]))+fpn_alt_outputs[-2]).shape

In [ ]:
alt_fpn.pan_layers_3x3

In [57]:
class YOLOHeads(nn.Module):
    def __init__(self, fpn_channels=64, num_classes=2, n_scales=3, **kwargs):
        super().__init__(**kwargs)
        self.fpn_channels = fpn_channels
        self.num_classes = num_classes
        self.n_scales = n_scales
        self.output_head_nodes = 4+1+self.num_classes

        self.head = nn.ModuleList([self._make_head(self.fpn_channels, self.output_head_nodes) for _ in range(self.n_scales)])

    @staticmethod
    def _make_head(fpn_channels, final_op_channels):
        layers = []

        for _ in range(4):
            layers.append(nn.Conv2d(fpn_channels, fpn_channels, kernel_size=3, stride=1, padding=1))
            layers.append(nn.ReLU(inplace=True))

        layers.append(nn.Conv2d(fpn_channels, final_op_channels, kernel_size=3, stride=1, padding=1))

        return nn.Sequential(*layers)

    def forward(self, feature_maps):

        preds = []

        for idx, feature_map in enumerate(feature_maps):
            pred = self.head[idx](feature_map)
            print(pred.shape)
            pred = pred.permute(0, 2, 3, 1).reshape(pred.shape[0], -1, self.output_head_nodes)
            preds.append(pred)

        preds = torch.cat(preds, dim=1)

        return preds

In [58]:
yolo_heads = YOLOHeads(fpn_channels=128, num_classes=2, n_scales=4)

In [59]:
yolo_preds = yolo_heads(fpn_alt_outputs)

torch.Size([1, 7, 75, 75])
torch.Size([1, 7, 38, 38])
torch.Size([1, 7, 19, 19])
torch.Size([1, 7, 10, 10])


In [60]:
yolo_preds.shape

torch.Size([1, 7530, 7])

In [ ]:
import cv2
from models.src.encoder import YOLODataEncoder
input_size=(300, 300)
encoder = YOLODataEncoder(input_size=input_size, strides=[4, 8, 16, 32])

In [ ]:
from models.utils.data import load_groundtruths

data_path = "dataset/Dataset/validation/Vehicle registration plate"
data = load_groundtruths(data_path, train=False, shuffle=False, debug=True)
images, boxes, labels, tot_snamples = data
idx = 1
sample = (images[idx], boxes[idx], labels[idx])
sample

In [ ]:
image = cv2.imread(sample[0])
img_size = image.shape
img_size

In [ ]:
bboxes = sample[1]
bboxes_norm = torch.tensor(bboxes, dtype=torch.float)

bboxes_norm[:, [0, 2]] *= (input_size[1] / img_size[1])
bboxes_norm[:, [1, 3]] *= (input_size[0] / img_size[0])

encoded = encoder.encode(bboxes_norm, sample[2])

In [ ]:
encoded.shape

In [ ]:
encoder.grid_centers.shape

In [ ]:
encoder.grid_centers

In [ ]:
encoder.grid_sizes

In [ ]:
encoder.decode(encoded)